# Rural Scene Understanding for Safe Parking

##Stage 1: Semantic Segmentation of Unstructured Parking Environments

## Overview

This project addresses the challenge of scene understanding in unstructured rural parking environments, where boundaries are ambiguous and layouts vary significantly. The objective was to perform semantic segmentation of three key classes: Drivable Area, Non-Drivable Area, and Vehicles, enabling structured interpretation of diverse rural parking scenes.

I trained a YOLOv8s segmentation model using the Ultralytics framework on a custom COCO-format dataset built from manually extracted dashcam footage and expanded through targeted augmentation to 1,469 annotated images. The final model achieved 0.728 mAP50–95 and 0.892 Mask mAP50 across all classes, demonstrating strong boundary precision and robust generalization under varied environmental conditions. Performance gains were achieved through iterative re-annotation, hyperparameter optimization, and systematic comparison of model variants.

## Dataset Access

The dataset used in this project is not publicly available due to privacy considerations (dashcam footage).

## Problem Definition

### Task  
The goal of this stage is to perform **semantic segmentation** of unstructured rural parking environments. The model identifies three key classes: **Drivable Area**, **Non-Drivable Area**, and **Vehicles**, forming the foundation for automated safe parking detection.

### Why it is challenging  
Rural parking areas are highly unstructured, with unclear boundaries and diverse layouts. Variations in terrain, lighting conditions, and occlusions from vegetation or vehicles make segmentation difficult. Additionally, dashcam footage introduces motion blur, varying viewpoints, and inconsistent image quality, requiring robust models and preprocessing.

### Why it matters
Accurate segmentation of drivable and non-drivable areas, along with vehicles, is essential for understanding and analyzing unstructured rural parking environments. This work demonstrates how computer vision can be applied to real-world dashcam data to identify safe parking areas, providing a foundation for future research or applications in driver assistance, traffic analysis, or automated parking tools.

## Dataset and Preprocessing

### Dataset Description
The dataset consists of **1,469 annotated images**, derived from dashcam videos that I personally recorded in rural parking areas. Three classes were labeled: **Drivable_area**, **Non_drivable**, and **Vehicles**. The dataset is split into **1,323 training images**, **118 validation images**, and **28 test images**. Augmentation techniques were applied to increase dataset diversity and improve model robustness to variations in lighting, terrain, and viewpoint.


To reduce dataset bias, the dataset intentionally includes frames without visible vehicles or clearly defined parking areas. These negative and low-activity samples prevent the model from over-associating rural scenes with vehicle presence and encourage more robust scene understanding across diverse environmental conditions.

Class Naming Note:

During the course of the project, the class originally labeled **Obstacle** was renamed to **Non_drivable** following refinement of annotation guidelines. Some earlier experiments and figures therefore reference Obstacle, while later results use Non_drivable. The two terms refer to the same semantic category, with improved boundary definition and labeling consistency introduced in the later phase.

### Frame Extraction

Frames were extracted every 15th frame to balance temporal diversity and dataset size. After extraction, redundant frames where the scene showed minimal variation were manually removed to reduce dataset bias and improve training efficiency.

In [ ]:
# VIDEO FRAME EXTRACTION SCRIPT

import cv2

# Configuration
VIDEO_INPUT_DIR = "PATH_TO_INPUT_VIDEOS"
OUTPUT_FRAMES_DIR = "PATH_TO_OUTPUT_FOLDER"
custom_prefix = "Location_4_" # Prefix added to all extracted frames to encode capture location and prevent filename overlap
# when aggregating frames from multiple recording sessions

os.makedirs(OUTPUT_FRAMES_DIR, exist_ok=True)

# Get all video files from input directory
video_files = [
    f for f in os.listdir(VIDEO_INPUT_DIR)
    if f.lower().endswith(('.mp4', '.mov', '.avi'))
]

# Extract 1 frame every 15 frames
for video_filename in video_files:
    video_path = os.path.join(VIDEO_INPUT_DIR, video_filename)
    cap = cv2.VideoCapture(video_path)

    frame_count, saved_count = 0, 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_count % 15 == 0:
            fname = f"{custom_prefix}{os.path.splitext(video_filename)[0]}_frame_{saved_count:05d}.jpg}"
            cv2.imwrite(os.path.join(OUTPUT_FRAMES_DIR, fname), frame)
            saved_count += 1

        frame_count += 1

    cap.release()

print("Frame extraction completed.")

#### Privacy Protection: License Plate Blurring
To ensure privacy, license plates and other sensitive information visible in the frames were automatically blurred using a YOLOv11 detection model. This allowed the dataset to be safely used for training without compromising privacy.

In [ ]:
#   LICENSE PLATE BLURRING SCRIPT

import os
import cv2
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
from tqdm import tqdm


# Configuration
FRAMES_INPUT_DIR = "PATH_TO_EXTRACTED_FRAMES"
CONF_THRESHOLD = 0.35

MODEL_REPO_ID = "morsetechlab/yolov11-license-plate-detection"
MODEL_FILENAME = "license-plate-finetune-v1x.pt"


# Load model
model_path = hf_hub_download(
    repo_id=MODEL_REPO_ID,
    filename=MODEL_FILENAME
)

yolo_plate = YOLO(model_path)


# Blur function
def blur_boxes(image, boxes, ksize=(51, 51), sigma=30):
    for (x1, y1, x2, y2) in boxes:
        x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
        roi = image[y1:y2, x1:x2]
        if roi.size > 0:
            image[y1:y2, x1:x2] = cv2.GaussianBlur(roi, ksize, sigma)
    return image


# Apply blurring
frame_files = sorted([
    f for f in os.listdir(FRAMES_INPUT_DIR)
    if f.lower().endswith(".jpg")
])

for fname in tqdm(frame_files, desc="Blurring license plates"):
    image_path = os.path.join(FRAMES_INPUT_DIR, fname)
    img = cv2.imread(image_path)

    if img is None:
        continue

    results = yolo_plate.predict(img, verbose=False)[0]

    boxes = [
        box.cpu().numpy()
        for box, conf in zip(results.boxes.xyxy, results.boxes.conf)
        if float(conf) >= CONF_THRESHOLD
    ]

    if boxes:
        img = blur_boxes(img, boxes)

    cv2.imwrite(image_path, img)

print("License plate blurring completed.")

#### Annotation Strategy

All frames were manually annotated with three semantic classes: `Drivable_area`, `Non_drivable`, and `Vehicle`, using Roboflow.

I defined the class taxonomy and annotation protocol to align the dataset with the project’s objective of identifying safe parking areas in unstructured rural environments. Special attention was given to boundary precision between drivable and non-drivable regions, as this distinction is critical for reliable segmentation.

During experimentation, annotation guidelines were refined and parts of the dataset were re-annotated to correct boundary inconsistencies and improve class clarity. These refinements resulted in measurable performance improvements, particularly in mask precision and boundary accuracy.

#### Data Augmentation
To increase dataset diversity and improve model generalization, augmentations were applied using Roboflow, which allowed seamless integration with the annotated dataset.

Each training image was augmented three times with transformations including:

- Horizontal flips

- Hue adjustments (-8° to +8°)

- Saturation adjustments (-12% to +12%)

- Brightness adjustments (-25% to +25%)

- Exposure adjustments (-12% to +12%)

These augmentations expanded the effective size of the dataset and helped the model handle diverse lighting and environmental conditions in rural parking scenes.

## Baseline Training Setup

### Initial Model Selection

For the baseline experiment, I used the YOLOv8 small segmentation model (yolov8s-seg) from Ultralytics.

The small (s) variant was selected because it offers:

- Good accuracy–speed trade-off

- Lower memory usage

- Faster experimentation cycles

- Suitability for potential real-time deployment

- Pretrained weights were used to leverage transfer learning.

### Baseline Hyperparameters

For the baseline training, I selected 50 epochs, image size 640, and batch size 16 to ensure a balance between fast experimentation and sufficient convergence. These parameters allow the model to learn initial segmentation patterns effectively while keeping GPU memory usage reasonable for a standard workstation or Colab environment.

In [ ]:
# BASELINE YOLOv8s SEGMENTATION TRAINING


# Install Ultralytics (if not already installed)
!pip install ultralytics --upgrade

from ultralytics import YOLO
import torch

# Configuration/Placeholders

DATA_YAML = "PATH_TO_DATASET_YAML"        # path to your dataset.yaml
PROJECT_DIR = "PATH_TO_OUTPUT_WEIGHTS"    # folder to save trained weights
MODEL_NAME = "baseline_seg_v1"            # name of this training run

EPOCHS = 50                               # initial number of training epochs
IMG_SIZE = 640                            # input image size
BATCH_SIZE = 16                           # batch size for training


# Check GPU
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Not available")

# Load pretrained YOLOv8s segmentation model
model = YOLO("yolov8s-seg.pt")


# Baseline training to establish initial model performance
# Chosen parameters balance speed of experimentation and sufficient convergence
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    name=MODEL_NAME,
    project=PROJECT_DIR,
    exist_ok=True,
    plots=True
)

## Baseline Evaluation

### Model Evaluation

During development, each model version was evaluated on the **validation set immediately after training**.  
This allowed quick identification of underperforming classes (e.g., the original `Obstacle` class, later renamed `Non_drivable`) and guided iterative improvements through **refined annotation rules** and **hyperparameter tuning**.  

Only after all refinements were complete was the final model evaluated on an **unseen test set** to obtain the true generalization performance.  
Validation metrics provided insight into class-wise performance and informed subsequent dataset and model improvements.

In [ ]:
# Metrics Summary

# Load trained model
model = YOLO("PATH_TO_MODEL_WEIGHTS")  # replace with your current trained version

# Evaluate on validation data
metrics = model.val(data="PATH_TO_DATASET_YAML", verbose=True)

# Print metrics summary
print("\nValidation Metrics Summary:")
print(metrics)

### Baseline Metrics

Segmentation Performance Summary (Mask Metrics)

| Class         | Instances | Precision | Recall | mAP0.50 | **mAP0.50-0.95** |
| ------------- | --------- | --------- | ------ | -------- | ----------------- |
| **Overall**   | 716       | 0.696     | 0.606  | 0.609    | **0.325**         |
| Drivable_area | 126       | 0.799     | 0.937  | 0.918    | **0.551**         |
| Obstacle*      | 348       | 0.604     | 0.236  | 0.268    | **0.071**         |
| Vehicle       | 242       | 0.687     | 0.645  | 0.640    | **0.353**         |
----------------------------
*class Obstacle was later renamed to Non_drivable

### Training Curves

In [ ]:
# Baseline Plots (paths to the saved images)


# Train vs Validation Loss

![Baseline Train vs Validation Loss](assets/plots/baseline_train_vs_val_loss.png)

# Validation mAP

![Baseline Validation Segmentation mAP](baseline_val_seg_map.png)

# Validation Precision & Recall

![Baseline Validation Precision and Recall](assets/plots/baseline_val_p_and_r.png)



In [ ]:
# SCRIPT FOR BASELINE YOLOv8 SEGMENTATION METRICS PLOTS

import os
import pandas as pd
import matplotlib.pyplot as plt

# ------------------ CONFIGURATION ------------------
RESULTS_CSV_PATH = "PATH_TO_RESULTS_CSV"   # Example: "./results/results.csv"
PLOT_OUTPUT_DIR = "PATH_TO_SAVE_PLOTS"     # Optional: save plots here
os.makedirs(PLOT_OUTPUT_DIR, exist_ok=True)

# ------------------ LOAD RESULTS CSV ------------------
df = pd.read_csv(RESULTS_CSV_PATH)
epochs = df["epoch"]

print("Columns found in results.csv:")
print(df.columns.tolist())

# ------------------ LOSS PLOTS ------------------
plt.figure(figsize=(12,6))

plt.plot(epochs, df.get("train/seg_loss", []), label="Train Seg Loss")
plt.plot(epochs, df.get("val/seg_loss", []), label="Val Seg Loss")

plt.plot(epochs, df.get("train/box_loss", []), "--", label="Train Box Loss")
plt.plot(epochs, df.get("val/box_loss", []), "--", label="Val Box Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss (Overfitting Check)")
plt.legend()
plt.grid(True)

# Save plot if output folder is set
loss_plot_path = os.path.join(PLOT_OUTPUT_DIR, "loss_plot.png")
plt.savefig(loss_plot_path, dpi=150)
plt.show()

# ------------------ mAP PLOTS ------------------
plt.figure(figsize=(12,6))

plt.plot(epochs, df.get("metrics/mAP50(M)", []), label="mAP50")
plt.plot(epochs, df.get("metrics/mAP50-95(M)", []), label="mAP50-95")

plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.title("Validation Segmentation mAP")
plt.legend()
plt.grid(True)

# Save plot
map_plot_path = os.path.join(PLOT_OUTPUT_DIR, "map_plot.png")
plt.savefig(map_plot_path, dpi=150)
plt.show()

# ------------------ PRECISION / RECALL ------------------
plt.figure(figsize=(12,6))

plt.plot(epochs, df.get("metrics/precision(M)", []), label="Precision")
plt.plot(epochs, df.get("metrics/recall(M)", []), label="Recall")

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Precision & Recall")
plt.legend()
plt.grid(True)

# Save plot
pr_plot_path = os.path.join(PLOT_OUTPUT_DIR, "precision_recall_plot.png")
plt.savefig(pr_plot_path, dpi=150)
plt.show()

print("All plots completed and saved to:", PLOT_OUTPUT_DIR)

### Technical Interpretation

The baseline YOLOv8s segmentation model achieves an overall mAP0.50-0.95 of 0.325, indicating moderate segmentation quality under strict IoU thresholds. The noticeable drop from mAP0.50 (0.609) to mAP0.50-0.95 (0.325) highlights limitations in boundary precision, particularly under higher overlap requirements.

Although the model performs relatively well on Drivable_area and demonstrates reasonable baseline performance on Vehicle, the Obstacle class shows consistently weak results, even at the relaxed threshold (mAP0.50 = 0.268).

Because the same architecture and training configuration perform substantially better on other classes, the results suggest that dataset and annotation strategy may be a major contributing factor to the poor Obstacle performance. However, architectural limitations and input resolution constraints remain potential factors and require controlled experimentation to isolate the root cause.

### Training Dynamics Assessment

Overfitting

Training losses decrease consistently, while validation segmentation loss begins increasing after ~epoch 20. This indicates moderate late-stage overfitting, primarily affecting mask refinement rather than overall detection quality.

Convergence

Validation mAP stabilizes around epochs 25–30, suggesting the model has converged. Additional epochs provide minimal performance gains.

Stability

Precision, recall, and mAP fluctuate within a narrow band after convergence, with no signs of training instability or metric collapse. Optimization appears stable and well-behaved.

Conclusion

Further improvements should focus on refining the dataset (annotation quality, class definitions, and selective re-labeling), experimenting with image resolution, and considering model architecture adjustments.

## Model Improvement and Iterative Experiments

To address observed performance limitations, initially most pronounced in the Obstacle class, I conducted iterative optimisation incorporating input resolution and model scale experiments, controlled re-annotation studies, and systematic hyperparameter tuning. This was followed by structured component-wise ablation and controlled rollback validation to assess the impact of each modification on mAP, convergence stability, and generalization.

### Annotation Refinement Iterations

#### 1. Full-Object Annotation Standardization for the Obstacle Class.

**What Was Changed**

Annotation guidelines for the Obstacle class were clarified to require labeling the entire object, rather than partial or approximate coverage.
The full dataset was re-annotated according to this updated rule.

**Why**

The previous strategy (annotating only part of the object) introduced subjectivity and inconsistency.
The definition of “half” was inherently ambiguous and dependent on the annotator’s judgment, leading to unstable instance boundaries and noisy supervision.

**Performance impact**

- Obstacle mAP0.50 increased from 0.268 to 0.6658

- This represents a **+148% improvement**, more than doubling baseline performance

- Other classes remained approximately at their initial performance levels

This strongly indicates that annotation consistency was the primary bottleneck for this class.

**Note**

The initial decision to annotate only the horizontal half of each Obstacle instance was motivated by concerns about class imbalance. In many images, Obstacle objects occupied a large portion of the frame, and fully annotating them was expected to disproportionately increase their pixel representation. This raised concerns that the model might bias toward the Obstacle class during training. However, subsequent results demonstrated that annotation consistency and complete object representation were more critical for stable learning than pixel distribution alone.

In [ ]:
# Illustions: Full-Object Annotation Standardization for the Obstacle Class

![Obstacle initial annotation](assets/examples/obstacle_half.png)
![obstacle refined annotation](assets/examples/obstacle_full.png)

#### 2. Removal of Small-Distant Vehicles.

**What Was Changed**

Small, distant vehicles were removed from the dataset annotations.

**Why**

With a maximum image resolution of 1024 px and training performed using YOLOv8m-seg (the largest model stably trainable within the constraints of the free tier of Google Colab), the model struggled to reliably detect very small objects.

At the same time, the primary goal of the project is safe parking area identification rather than long-range vehicle detection. Distant vehicles do not influence immediate parking decisions and therefore do not provide task-relevant information. Their inclusion increased object-scale variance and detection difficulty without improving practical system performance, making their removal both technically justified and aligned with the project’s functional scope.

**Performance impact**

- Vehicle mAP0.50 improved from 0.640 to 0.830

- Approximately **+30% performance gain**

This breakthrough highlighted a critical scale mismatch between annotation and training resolution. While annotations were created on high-resolution (3840×2160) images with zoom capability, the model was trained on downscaled inputs (640×640 / 1024×1024), where distant vehicles became nearly indistinguishable. Aligning dataset annotations with the model’s effective visual resolution eliminated this perceptual gap and directly improved Vehicle class performance.

In [ ]:
# Illustions: Removal of Small-Distant Vehicles

![Small Vehicle removal](assets/examples/vehicle_improvement.png)

#### 3. One-Mask-Per-Object Policy for Non_drivable class.

**What Was Changed**

The Non_drivable class was re-annotated to enforce a strict one-mask-per-object policy.

**Why**

The previous multi-mask strategy introduced instance ambiguity and label fragmentation.
This negatively affected:

IoU alignment

Mask consistency

Instance separation

Fragmented masks made it difficult for the model to learn coherent object representations.

**Performance impact**

Non_drivable mAP0.50 increased by approximately +47%

Final mAP0.50 reached 0.976

This demonstrates that annotation structure directly influences segmentation quality. Standardizing the annotation protocol reduced label fragmentation and ambiguity, improved instance definition, and resulted in significantly higher mAP.


In [ ]:
# Illustions: One Mask per Object Policy for Non_drivable

![Multible Masks per Object](assets/examples/multiple_masks_per_object.png)
![One Mask per Object](assets/examples/one_mask_per_object.png)

#### 4. Improved Functional Scene Interpretation

**Enhanced Physical Accessibility Reasoning**

After improving annotation consistency for the Non_drivable class, the model demonstrated improved understanding of scene logic rather than relying solely on visual texture or color cues. For example, areas that were visually open (appearing drivable) but physically unreachable (e.g., located behind fences or obstacles) were correctly predicted as non-drivable. Initially, the model classified these regions as Drivable_area.

In [ ]:
# Illustions: Improved Understanding of Scene Logic

![Area Behind Fences Before Improving Consistency](assets/examples/behind_fence_before.png)
![Area Behind Fences After Improving Consistency](assets/examples/behind_fence_after.png)

**Learning Curb-Height as an Operational Boundary Cue**

The improvements in annotation consistency also allowed the model to learn to use curb height as a functional boundary cue. Sidewalks with a high vertical edge were classified as Non_drivable, while low-edge sidewalks were predicted as Drivable, consistent with the annotation policy designed for unstructured rural environments. This indicates that the model captured structural features linked to practical vehicle accessibility rather than relying solely on surface texture.

In [ ]:
# Illustions: Curb Size Recognision

![Curb Recognision mistake](assets/examples/curb_before.png)
![Curb Recognision correct](assets/examples/curb_after.png)

### Hyperparameter Tuning

Hyperparameter tuning was performed iteratively alongside dataset improvements to understand how training configuration affects segmentation performance. The main parameters explored were model size, input resolution, batch size, and training epochs. Most of these parameters produced only minor changes in segmentation performance. In contrast, input resolution had a noticeable impact, with larger image sizes improving segmentation quality, particularly for small or distant objects.

#### Baseline

The initial baseline udes:


*   Model: YOLOv8s-seg
*   Image size: 640x640
*   Batch size: 16





#### Batch Size and Learning Rate

Experiments were conducted with barch sizes 2, 4 and 16 and learning rates around 0.002 and 0.0025.

The motivation was:
- the dataset was relatively small
- masks varied significantly in size

There changes produced only marginal improvements, indicating that training dynamics were not the main bottleneck.

#### Model Capacity

Both YOLOv8s-seg and YOLOv8m-seg were tested.

Observations:
- Early experiments suggested that the larger model (YOLOv8m) performed slightly better.
- However, once annotation quality improved, image resolution increased, and noisy labels were removed, the smaller model (YOLOv8s) achieved equal or better performance.

This suggests that improved dataset clarity reduced the need for higher model capacity.

#### Image Resonlution

Increasing input resolution from 640x640 to 1024x1024 produced a noticeable improvement:
- Vehicle mAP50 increased by ~12%
- Non_Drivable mAP50 improved by ~8%

Higher resolution helped the model better detect:
- small vehicles
- object boundaries

Therefore, 1024x1024 was used for subsequent experiments.

#### Epochs and Overfitting Control

Longer training (up to 300 epochs) produced strong results but introduced overfitting after ~60 epochs.

To address this:
- model was changed to YOLOv8s-seg (smaller model has fewer parameters, therefore, less risk of overfitting, better for small datasets)
- training was reduced to 70 epochs
- save_period=1 was enabled, allowing manual inspection and selection of the most suitable checkpoint.

This allowed maintaining high performance while avoiding overfitting.

In [ ]:
# Illustion: Overfitting Check (YOLOv8m-seg)

![Train vs Validation Loss](assets/plots/overfitting_check_yolov8m.png)

In [ ]:
# Illustion: Overfitting Check (YOLOv8s-seg)

![Train vs Validation Loss](assets/plots/overfitting_check_yolov8s.png)


### Result

The experiments showed that dataset quality and annotation consistency had a significantly larger impact than hyperparameter tuning. Once noisy labels (e.g., small vehicles) were removed and masks were standardized (one mask per object), even a lighter model achieved strong performance.

This suggests that careful dataset curation can be more important than model or training configuration adjustments when working with relatively small datasets.

## Final Model Performance

Overall Performance:

Significant improvement from baseline; Drivable_area and Non_drivable achieve near-perfect segmentation (mAP50 > 0.97).

Vehicle Class:

Remains the most challenging class on test (mAP50 ~0.70) due to occlusion, scale variability, and limited test instances.

Precision / Recall:

High overall precision (0.89) and recall (0.86); Non_drivable recall reaches 1.0.

Model Comparison:

YOLOv8s-seg (small) performs nearly as well as YOLOv8m-seg (medium); smaller model is sufficient with clean, consistent data.

Key Insight for Vehicles:

Training on full-resolution images would allow annotation of all instances, near and far, reducing model confusion and improving detection across scales.

| Class             | Baseline mAP50 / mAP50–95 | YOLOv8m-seg Val | YOLOv8s-seg Val | YOLOv8s-seg Test |
| ----------------- | ------------------------- | --------------- | --------------- | ---------------- |
| **Drivable_area** | 0.918 / 0.551             | 0.994 / 0.825   | 0.995 / 0.825   | 0.979 / 0.825    |
| **Non_drivable**  | 0.268 / 0.071             | 0.976 / 0.843   | 0.985 / 0.843   | 0.995 / 0.843    |
| **Vehicle**       | 0.640 / 0.353             | 0.824 / 0.517   | 0.861 / 0.517   | 0.701 / 0.517    |
| **Average**       | 0.609 / 0.325             | 0.931 / 0.728   | 0.947 / 0.728   | 0.892 / 0.728    |


### Final Hyperparameter Configuration

- Model: YOLOv8s-seg
- Image size: 1024x1024
- Batch size: 2
- Learning rate: 0.002
- Epochs: 70
- Optimizer: Adam

Overall Performance:

- Drivable_area and Non_drivable achieve near-perfect segmentation (mAP50 > 0.97)
- Vehicle Class: Most challenging due to occlusions and scale variability (mAP50 ~0.70)
- Precision / Recall: High overall precision (0.89) and recall (0.86)
- Model Comparison: YOLOv8s-seg performs nearly as well as YOLOv8m-seg
- Key Insight for Vehicles: Full-resolution training would improve Vehicle detection across scales

In [ ]:
# Visual Predictions on Test Images

![Test Predictions](assets/examples/test_set.png)

## Conclusion


#### What Worked

This project demonstrates that reliable scene understanding in unstructured rural parking environments can be achieved using semantic segmentation with a relatively lightweight model when the dataset is carefully curated.

The final YOLOv8s-seg model achieved:

0.728 mAP50–95

0.892 Mask mAP50

with near-perfect segmentation for Drivable_area and Non_drivable classes and strong performance on Vehicle despite scale variability and occlusions.

A key outcome of the project is that strong performance was obtained without requiring a large model. After dataset refinement, the smaller YOLOv8s-seg model performed nearly as well as the larger YOLOv8m-seg, confirming that efficient models can achieve high-quality segmentation when trained on consistent data.

#### What Mattered Most

Although extensive hyperparameter tuning was performed (model scale, batch size, learning rate, epochs), most configuration changes produced only minor performance differences.

The largest improvements came from dataset and annotation refinement:

Standardizing full-object annotations for the Non_drivable class

Removing small distant vehicles that were not visible at training resolution

Enforcing a one-mask-per-object annotation policy

These changes significantly improved segmentation stability and mask quality. For example, the Non_drivable class improved from 0.268 mAP50 in the baseline to 0.976 mAP50 after annotation refinement.

This demonstrates an important practical insight:

For small to medium datasets, dataset quality and annotation consistency often matter more than hyperparameter tuning.

#### What I Learned

Several key lessons emerged during this project:

1. Annotation quality directly affects model performance

Inconsistent or fragmented annotations introduce noise that prevents the model from learning stable object representations.

2. Annotation scale must match model resolution

Annotating objects that are smaller than what the model can realistically resolve during training leads to unstable learning. Aligning annotation decisions with the model’s effective visual resolution greatly improves results.

3. Dataset design influences model reasoning

After improving annotation consistency, the model began learning functional scene cues (such as curb height or physical accessibility) rather than relying purely on visual texture or color.

4. Clean data can reduce the need for larger models

Once the dataset was refined, a smaller architecture performed almost as well as a larger one, suggesting that model capacity was not the main limitation.

Future Improvements

The main limitation of the current system is the training resolution constraint imposed by available GPU resources.

The original images were recorded at 3840×2160, but training was performed at 1024×1024. Because of this downscaling, very small vehicles had to be removed from annotations to avoid introducing noise.

With access to more powerful GPUs, several improvements could be explored:

Training with higher input resolution

Reintroducing small distant vehicles

Allowing the model to learn multi-scale vehicle detection

Training closer to the original image resolution would reduce the need to estimate which objects the model can realistically perceive and would allow the model to learn from all visible scene elements.